# Big Bang nucleosynthesis

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/big_bang_nucleosynthesis)

The first three minutes as a stiff reaction network: neutrons and protons freeze out of weak equilibrium, the deuterium bottleneck opens near $T \approx 80$ keV and almost every neutron ends up in $^4$He. Twelve reactions link n, p, d, t, $^3$He, $^4$He, $^7$Li and $^7$Be; a whole batch of baryon-to-photon ratios $\eta$ is integrated at once with an implicit BDF2 scheme, giving the Schramm plot. At the Planck value $\eta \approx 6.1\times10^{-10}$ helium and deuterium agree with observation, while lithium comes out about 3× too high: the cosmological lithium problem, still unresolved.

**Vector form**

$$\frac{dY_i}{dt} = \sum_r (P_{ri} - R_{ri})\Big(\lambda^+_r \prod_j Y_j^{R_{rj}} - \lambda^-_r \prod_j Y_j^{P_{rj}}\Big),\qquad \lambda^-_r = \lambda^+_r\, n_b^{-\Delta N_r} \prod_i (g_i n_{Q,i})^{R_{ri}-P_{ri}}\, e^{-Q_r/T},\qquad H^2 = \frac{8\pi}{3M_P^2}\big(\rho_\gamma + \rho_{e^\pm} + \rho_\nu\big)$$

**Index (tensor) form**

$$\frac{dY_{bi}}{du} = \frac{dt}{du}\,S_{ri}\big(\lambda^+_{br} M^+_{br} - \lambda^-_{br} M^-_{br}\big),\qquad M^\pm_{br} = \prod_j Y_{bj}^{E^\pm_{rj}},\qquad \Big(\delta_{ik} - \beta h\,\frac{dt}{du}\,J_{bik}\Big)\,\Delta Y_{bk} = -G_{bi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $R_{ri},\ P_{ri}$ | `stoichiometry(device)` | (Rx, N) | how many nuclei $i$ reaction $r$ consumes and produces (12 reactions, 8 species) |
| $S_{ri}$ | `P - R` | (Rx, N) | net change; $\sum_i A_i S_{ri} = 0$ for every $r$ (baryon number) |
| $N_A\langle\sigma v\rangle_r$ | `exp(einsum('sk,tk->ts', a, phi)).index_add` | (T, Rx) | REACLIB fits: repeated $k$ (7 basis functions of $T_9$) ⇒ summed, then sets of one reaction added |
| $\lambda^\pm_{br}$ | `rate_coefficients(T, eta, tau_n)` | (T, B, Rx) | forward ($\propto n_b \propto \eta$) and reverse (detailed balance) rates for every $\eta_b$ |
| $M^\pm_{br}$ | `monomials(Y, R)` | (B, Rx) | $\prod_j Y_j^{R_{rj}}$, e.g. $Y_d^2$ for d + d |
| $S_{ri}(\dots)_{br}$ | `einsum('ri,br->bi', S, flux)` | (B, N) | repeated $r$ ⇒ summed over reactions |
| $J_{bik}$ | `einsum('ri,brk->bik', S, dflux)` | (B, N, N) | exact Jacobian for Newton's method |
| $\Delta Y_{bk}$ | `torch.linalg.solve(eye - k * J, G)` | (B, N) | one Newton update of the implicit BDF2 step for the whole $\eta$ batch |
| $\lambda_{n\to p}$ | `weak_rates(T, T_nu, tau_n)` | (T,) | Born-approximation weak rates, normalised to the neutron lifetime $\tau_n$ |

Status: standard Big Bang nucleosynthesis is established physics. Its predictions for $^4$He and D agree with observations at the baryon density measured independently from the cosmic microwave background, one of the pillars of the hot Big Bang model (Alpher, Bethe and Gamow 1948; Wagoner, Fowler and Hoyle 1967). The cosmological lithium problem is open: the predicted $^7$Li/H is about 3× the Spite plateau of old halo stars ($(1.6\pm0.3)\times10^{-10}$), and neither nuclear physics, stellar depletion nor new particles has been shown to explain it. Observed values (cited constants): $Y_p = 0.245\pm0.003$ (PDG review of BBN), D/H $= (2.527\pm0.030)\times10^{-5}$ (Cooke, Pettini and Steidel 2018). Model: the 12 reactions of Smith, Kawano and Malaney (1993) with thermonuclear rates from the JINA REACLIB database (Cyburt et al. 2010; sets an06, de04, gi17, go17, cd08, db18) and reverse rates from detailed balance. Approximations, stated: Born-approximation weak rates (no radiative, finite-mass or finite-temperature corrections), instantaneous neutrino decoupling ($N_\mathrm{eff} = N_\nu$ instead of 3.044), $^7$Be counted as $^7$Li and t as $^3$He (later decays), and the T9 fits held constant above $T_9 = 10$. These make $Y_p$ about 2% lower than precision codes such as PRIMAT or PArthENoPE ($0.2470$); D/H agrees to a few per cent. Try $N_\nu = 4$: faster expansion freezes out more neutrons and raises $Y_p$ by about 0.013, which is how BBN counts neutrino species.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/big_bang_nucleosynthesis/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    eta_mark = params["eta10"] * 1e-10
    etas = tensors.eta_batch(params["n_eta"], eta_mark, dev)  # (B,) the whole Schramm plot in one batch
    res = solve.run_bbn(etas, params["tau_n"], params["n_nu"], params["n_steps"], dev)

    print(
        f"{len(etas)} values of η integrated together, {params['n_steps']} implicit BDF2 steps from T = 3 MeV to 1 keV"
    )
    print(f"τ_n = {params['tau_n']} s, N_ν = {params['n_nu']}")
    Y = res["Y"][:, -1]
    for T_show in (1.0, 0.25, 0.08):
        k = int((res["T"] - T_show).abs().argmin())
        print(f"T = {T_show:5.2f} MeV  t = {float(res['t'][k]):7.1f} s   n/p = 1/{float(Y[k, 1] / Y[k, 0]):.2f}")
    print(f"{'η × 10¹⁰':>9} {'Y_p':>8} {'D/H':>10} {'³He/H':>10} {'⁷Li/H':>10}")
    for b in range(len(etas)):
        print(f"{float(etas[b]) * 1e10:9.2f} {float(res['Yp'][b]):8.4f} {float(res['DH'][b]):10.3e}"
              f" {float(res['He3H'][b]):10.3e} {float(res['LiH'][b]):10.3e}")  # fmt: skip
    obs = tensors.OBSERVED
    print(f"at η = {params['eta10']:.2f} × 10⁻¹⁰:")
    print(f"  Y_p   = {float(res['Yp'][-1]):.4f}   observed {obs['Yp'][0]} ± {obs['Yp'][1]} (PDG)")
    print(f"  D/H   = {float(res['DH'][-1]):.3e}  observed (2.527 ± 0.030)e-5 (Cooke et al. 2018)")
    li = float(res["LiH"][-1])
    print(f"  ⁷Li/H = {li:.2e}   observed (1.6 ± 0.3)e-10 (Spite plateau): {li / obs['LiH'][0]:.1f}× too high,"
          " the cosmological lithium problem (unresolved)")  # fmt: skip
    return plot.animate(res, tensors.ETA_PLANCK, out_dir / "big_bang_nucleosynthesis.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/big_bang_nucleosynthesis/equation.py
r"""Physics → tensor translation: the thermal history and the nuclear reaction network.

Abundances are counted per baryon, Y_i = n_i / n_b, so expansion alone does not change them. Every reaction r
turns R_{ri} nuclei of species i into P_{ri} nuclei; with forward and reverse rate coefficients λ⁺_r, λ⁻_r

    dY_i/dt = Σ_r (P_{ri} − R_{ri}) ( λ⁺_r Π_j Y_j^{R_{rj}} − λ⁻_r Π_j Y_j^{P_{rj}} )        (r and j summed)

Baryon number is conserved because Σ_i A_i (P_{ri} − R_{ri}) = 0 for every r. The forward coefficient of a
two-body reaction is λ⁺ = n_b ⟨σv⟩ / (identical-particle factor), the reverse one follows from the Saha equation
(detailed balance):

    λ⁻_r = λ⁺_r · n_b^{−ΔN_r} · Π_i (g_i n_{Q,i})^{R_{ri} − P_{ri}} · e^{−Q_r/T},   n_Q = (m T / 2π)^{3/2}

so photodisintegration (ΔN = 1) does not care about η while fusion grows with n_b ∝ η: that is how η enters.

Clock: radiation domination, H² = (8π/3M_P²)(ρ_γ + ρ_e± + ρ_ν), and entropy conservation a³ T³ S(T) = const with
S = 2 + (7/2) h(m_e/T) (photons + e±). When e± annihilate, S falls from 11/2 to 2 and heats the photons, so
T_ν = T (S / (11/2))^{1/3} → (4/11)^{1/3} T. The integration variable is u = −ln T.
"""

import math

import torch

from .tensors import (
    CHARGE,
    HBAR,
    HBARC,
    M_E,
    M_PLANCK,
    M_U,
    MASS_EXCESS,
    MASS_NUMBER,
    MEV_PER_T9,
    N_AVOGADRO,
    Q_NP,
    SPIN_STATES,
    ZETA3,
    q_values,
    reaclib_table,
    stoichiometry,
)


# ---------------------------------------------------------------- thermal history
def electron_entropy_fraction(T: torch.Tensor, n_y: int = 400) -> tuple[torch.Tensor, torch.Tensor]:
    r"""h(z) = (ρ+P)_{e±} / (ρ+P)_{massless e±} and ρ_{e±} (MeV⁴), z = m_e/T. Momentum integral on y = p/T ∈ [0, 40].

    ρ = (4/2π²) T⁴ ∫ y² ε /(e^ε + 1) dy,   P = (4/2π²) T⁴ ∫ y⁴/(3ε) /(e^ε + 1) dy,   ε = √(y² + z²)
    """
    y = torch.linspace(0, 40, n_y, dtype=T.dtype, device=T.device)  # (Y,)
    z = (M_E / T)[:, None]  # (S, 1)
    eps = torch.sqrt(y**2 + z**2)  # (S, Y)
    occ = torch.sigmoid(-eps)  # Fermi–Dirac 1/(e^ε + 1)
    rho = torch.trapezoid(y**2 * eps * occ, y, dim=-1) * 4 / (2 * math.pi**2) * T**4
    P = torch.trapezoid(y**4 / (3 * eps) * occ, y, dim=-1) * 4 / (2 * math.pi**2) * T**4
    h = (rho + P) / (7 * math.pi**2 / 45 * T**4)
    return h, rho


def entropy_dof(T: torch.Tensor) -> torch.Tensor:
    """S(T) = 2 + (7/2) h: photon + e± entropy in units of (2π²/45) T³."""
    return 2 + 3.5 * electron_entropy_fraction(T)[0]


def neutrino_temperature(T: torch.Tensor) -> torch.Tensor:
    """T_ν = T (S/(11/2))^{1/3}: neutrinos decoupled before e± annihilation (instantaneous decoupling)."""
    return T * (entropy_dof(T) / 5.5) ** (1 / 3)


def hubble_rate(T: torch.Tensor, n_nu: float) -> torch.Tensor:
    r"""H = √(8πρ/3) / M_P in s⁻¹, ρ = π²T⁴/15 + ρ_e± + N_ν (7/8)(π²/15) T_ν⁴."""
    rho = math.pi**2 / 15 * T**4 + electron_entropy_fraction(T)[1]
    rho = rho + n_nu * 7 / 8 * math.pi**2 / 15 * neutrino_temperature(T) ** 4
    return torch.sqrt(8 * math.pi * rho / 3) / M_PLANCK / HBAR


def dt_du(T: torch.Tensor, n_nu: float, d: float = 1e-3) -> torch.Tensor:
    r"""dt/du with u = −ln T: (d ln a/du) / H, d ln a/du = 1 + ⅓ d ln S/d ln T (a ∝ 1/(T S^{1/3}))."""
    dlnS = (torch.log(entropy_dof(T * math.exp(d))) - torch.log(entropy_dof(T * math.exp(-d)))) / (2 * d)
    return (1 + dlnS / 3) / hubble_rate(T, n_nu)


def time_of(T: torch.Tensor, n_nu: float, T_hot: float = 100.0) -> torch.Tensor:
    """Cosmic time t(T) in s: t = 1/(2H) at T_hot, then t = ∫ dt/du du (trapezoid on a fine log grid)."""
    u = torch.linspace(-math.log(T_hot), -math.log(float(T.min())) + 1e-9, 3000, dtype=T.dtype, device=T.device)
    Tg = torch.exp(-u)
    rate = dt_du(Tg, n_nu)
    t0 = 0.5 / hubble_rate(Tg[:1], n_nu)
    t_grid = torch.cat([t0, t0 + torch.cumsum(0.5 * (rate[1:] + rate[:-1]) * u.diff(), 0)])
    k = torch.searchsorted(u, -torch.log(T)).clamp(1, len(u) - 1)
    w = (-torch.log(T) - u[k - 1]) / (u[k] - u[k - 1])
    return t_grid[k - 1] * (1 - w) + t_grid[k] * w


def baryon_density(T: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    """n_b in cm⁻³, (S, B): η n_γ(today-normalised) = η (2ζ(3)/π²) T³ · S/2 (photons heated later by e± entropy)."""
    n_gamma = 2 * ZETA3 / math.pi**2 * T**3 * entropy_dof(T) / 2 / HBARC**3  # (S,)
    return torch.einsum("s,b->sb", n_gamma, eta)


# ---------------------------------------------------------------- weak n ↔ p rates
def weak_rates(T: torch.Tensor, T_nu: torch.Tensor, tau_n: float, n_w: int = 4000) -> tuple[torch.Tensor, torch.Tensor]:
    r"""λ_{n→p}, λ_{p→n} in s⁻¹ (Born approximation, Weinberg 1972; form of Kawano 1992), (S,) each.

    λ_{p→n} = K ∫₁^∞ x√(x²−1) [ (x+q)² / ((1+e^{−xz})(1+e^{(x+q)z_ν})) + (x−q)² / ((1+e^{xz})(1+e^{−(x−q)z_ν})) ] dx
    λ_{n→p} = the same with q → −q.   x = E_e/m_e, q = Q/m_e, z = m_e/T, z_ν = m_e/T_ν, K fixed by λ_{n→p}(T→0) = 1/τ_n.
    Substituting x = 1 + w² removes the square-root edge at x = 1.
    """
    q = Q_NP / M_E
    z, zn = (M_E / T)[:, None], (M_E / T_nu)[:, None]  # (S, 1)
    w_max = torch.sqrt(q + 60 * T / M_E)[:, None]  # covers e^{−60} tails, (S, 1)
    w = torch.linspace(0, 1, n_w, dtype=T.dtype, device=T.device) * w_max  # (S, W)
    x = 1 + w**2
    jac = x * w * torch.sqrt(w**2 + 2) * 2 * w  # x √(x²−1) dx/dw

    def rate(qs: float) -> torch.Tensor:
        a = (x + qs) ** 2 * torch.sigmoid(x * z) * torch.sigmoid(-(x + qs) * zn)
        b = (x - qs) ** 2 * torch.sigmoid(-x * z) * torch.sigmoid((x - qs) * zn)
        return torch.trapezoid(jac * (a + b), w, dim=-1)

    w0 = torch.linspace(0, math.sqrt(q - 1), 20001, dtype=T.dtype, device=T.device)
    x0 = 1 + w0**2
    decay = torch.trapezoid(x0 * (x0 - q) ** 2 * w0 * torch.sqrt(w0**2 + 2) * 2 * w0, w0)  # ∫₁^q x(x−q)²√(x²−1)
    K = 1 / (tau_n * decay)
    return K * rate(-q), K * rate(q)


# ---------------------------------------------------------------- nuclear rates
def reaclib_rates(T: torch.Tensor) -> torch.Tensor:
    r"""N_A⟨σv⟩ (cm³ mol⁻¹ s⁻¹) of every reaction, (S, Rx); column 0 (weak) is 0.

    Σ over fit sets s: exp(a_{sk} φ_k(T9)), φ = (1, T9⁻¹, T9^{−1/3}, T9^{1/3}, T9, T9^{5/3}, ln T9): repeated k summed.
    The fits are used up to T9 = 10 (held constant above, where nuclei are photodissociated anyway).
    """
    a, idx = reaclib_table(T.device)
    T9 = (T / MEV_PER_T9).clamp(max=10.0)
    phi = torch.stack(
        [torch.ones_like(T9), 1 / T9, T9 ** (-1 / 3), T9 ** (1 / 3), T9, T9 ** (5 / 3), torch.log(T9)], -1
    )
    per_set = torch.exp(torch.einsum("sk,tk->ts", a, phi))  # (S_T, sets)
    out = torch.zeros(len(T), int(idx.max()) + 1, dtype=T.dtype, device=T.device)
    return out.index_add(1, idx, per_set)


def nuclear_masses(device: torch.device) -> torch.Tensor:
    """m_i = A_i m_u + Δ_i − Z_i m_e (MeV): nuclear masses from atomic mass excesses, (N,)."""
    return (MASS_NUMBER * M_U + MASS_EXCESS - CHARGE * M_E).to(device)


def reverse_coefficient(T: torch.Tensor, device: torch.device) -> torch.Tensor:
    r"""C_r(T) = Π_i (g_i n_{Q,i})^{R_{ri} − P_{ri}}, n_Q = (m T/2π)^{3/2} in cm⁻³, (S, Rx).

    Repeated i is summed in the logarithm."""
    R, P = stoichiometry(device)
    m = nuclear_masses(device)
    log_gnq = torch.log(SPIN_STATES.to(device)) + 1.5 * torch.log(torch.einsum("s,i->si", T, m) / (2 * math.pi))
    log_gnq = log_gnq - 3 * math.log(HBARC)  # (S, N)
    return torch.exp(torch.einsum("si,ri->sr", log_gnq, R - P))


def rate_coefficients(T: torch.Tensor, eta: torch.Tensor, tau_n: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""λ⁺_{sbr}, λ⁻_{sbr} in s⁻¹ on the temperature grid s for every η_b, (S, B, Rx) each."""
    dev = T.device
    R, P = stoichiometry(dev)
    Q = q_values(dev)
    n_b = baryon_density(T, eta)[:, :, None]  # (S, B, 1)
    n_in = R.sum(1)  # reactants per reaction
    identical = torch.exp(torch.lgamma(R + 1).sum(1))  # Π_i R_ri!: 2 for d + d (identical particles)
    sigv = reaclib_rates(T)[:, None, :] / N_AVOGADRO  # ⟨σv⟩ in cm³/s, (S, 1, Rx)
    lam_f = n_b ** (n_in - 1) * sigv / identical
    dN = n_in - P.sum(1)  # ΔN_r = Σ R − Σ P (1 for radiative captures)
    boltz = torch.exp(-Q / T[:, None])[:, None, :]  # (S, 1, Rx)
    lam_b = lam_f * n_b ** (-dN) * reverse_coefficient(T, dev)[:, None, :] * boltz
    lam_np, lam_pn = weak_rates(T, neutrino_temperature(T), tau_n)
    lam_f[..., 0] = lam_np[:, None]
    lam_b[..., 0] = lam_pn[:, None]
    return lam_f, lam_b


# ---------------------------------------------------------------- network
def monomials(Y: torch.Tensor, E: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""M_{br} = Π_i Y_{bi}^{E_ri} and its gradient G_{brk} = ∂M_{br}/∂Y_{bk}.   Y: (B, N), E: (Rx, N)."""
    base = Y[:, None, :] ** E  # (B, Rx, N)
    M = base.prod(-1)
    eye = torch.eye(Y.shape[1], dtype=torch.bool, device=Y.device)
    others = torch.where(eye, torch.ones_like(base)[:, :, :, None], base[:, :, None, :]).prod(-1)  # Π_{j≠k}
    G = E * Y[:, None, :] ** (E - 1).clamp(min=0) * others  # (B, Rx, N)
    return M, G


def network(Y: torch.Tensor, lam_f: torch.Tensor, lam_b: torch.Tensor, R: torch.Tensor, P: torch.Tensor):
    r"""dY_{bi}/dt = Σ_r (P−R)_{ri} (λ⁺_{br} M⁺_{br} − λ⁻_{br} M⁻_{br}) and its Jacobian J_{bik}: (B, N), (B, N, N)."""
    Mf, Gf = monomials(Y, R)
    Mb, Gb = monomials(Y, P)
    S = P - R
    f = torch.einsum("ri,br->bi", S, lam_f * Mf - lam_b * Mb)
    J = torch.einsum("ri,brk->bik", S, lam_f[..., None] * Gf - lam_b[..., None] * Gb)
    return f, J


# ---------------------------------------------------------------- observables
def observables(Y: torch.Tensor) -> dict:
    """Y_p = 4 Y_⁴He (helium mass fraction) and number ratios to hydrogen; t → ³He and ⁷Be → ⁷Li decay later."""
    H = Y[..., 1]
    return {
        "Yp": 4 * Y[..., 5],
        "DH": Y[..., 2] / H,
        "He3H": (Y[..., 4] + Y[..., 3]) / H,
        "LiH": (Y[..., 6] + Y[..., 7]) / H,
    }


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/big_bang_nucleosynthesis/tensors.py
"""Build the concrete tensors: species, the reaction network, rate-fit coefficients, the η batch and the T grid.

Units: MeV for temperatures and energies, seconds, cm⁻³ for number densities. float64 throughout (the network is
stiff and abundances span 30 orders of magnitude).
"""

import math

import torch

F64 = torch.float64

# physical constants (CODATA 2018 / PDG)
HBAR = 6.582119569e-22  # MeV s
HBARC = 1.973269804e-11  # MeV cm
M_PLANCK = 1.220890e22  # MeV, G = 1 / M_P²
M_E = 0.51099895  # MeV, electron mass
M_U = 931.49410242  # MeV, atomic mass unit
Q_NP = 1.29333236  # MeV, m_n − m_p
N_AVOGADRO = 6.02214076e23
ZETA3 = 1.2020569031595942
MEV_PER_T9 = 0.08617333262  # k_B × 10⁹ K in MeV

# species i: neutron, proton, deuteron, triton, helium-3, helium-4, lithium-7, beryllium-7
SPECIES = ["n", "p", "d", "t", "³He", "⁴He", "⁷Li", "⁷Be"]
MASS_NUMBER = torch.tensor([1, 1, 2, 3, 3, 4, 7, 7], dtype=F64)  # A_i
CHARGE = torch.tensor([0, 1, 1, 1, 2, 2, 3, 4], dtype=F64)  # Z_i
SPIN_STATES = torch.tensor([2, 2, 3, 2, 2, 1, 4, 4], dtype=F64)  # g_i = 2J + 1
MASS_EXCESS = torch.tensor(  # atomic mass excess Δ_i in MeV (AME 2020); Q = Σ Δ(reactants) − Σ Δ(products)
    [8.0713181, 7.2889711, 13.1357220, 14.9498060, 14.9312155, 2.4249159, 14.9071048, 15.7689990], dtype=F64
)

# the 12 reactions that matter for A ≤ 7 (Smith, Kawano & Malaney 1993): reactants → products (photons not listed)
REACTIONS = [
    (["n"], ["p"]),  # 0  weak: n + ν ↔ p + e⁻, n + e⁺ ↔ p + ν̄, n ↔ p + e⁻ + ν̄
    (["p", "n"], ["d"]),  # 1  p(n,γ)d
    (["d", "p"], ["³He"]),  # 2  d(p,γ)³He
    (["d", "d"], ["³He", "n"]),  # 3  d(d,n)³He
    (["d", "d"], ["t", "p"]),  # 4  d(d,p)t
    (["³He", "n"], ["t", "p"]),  # 5  ³He(n,p)t
    (["t", "d"], ["⁴He", "n"]),  # 6  t(d,n)⁴He
    (["³He", "d"], ["⁴He", "p"]),  # 7  ³He(d,p)⁴He
    (["³He", "⁴He"], ["⁷Be"]),  # 8  ³He(α,γ)⁷Be
    (["t", "⁴He"], ["⁷Li"]),  # 9  t(α,γ)⁷Li
    (["⁷Be", "n"], ["⁷Li", "p"]),  # 10 ⁷Be(n,p)⁷Li
    (["⁷Li", "p"], ["⁴He", "⁴He"]),  # 11 ⁷Li(p,α)⁴He
]

# JINA REACLIB fits (Cyburt et al. 2010, ApJS 189, 240; snapshot "default2" 2025-03-30 as shipped with pynucastro):
# N_A⟨σv⟩ = Σ_sets exp(a0 + a1/T9 + a2 T9^{-1/3} + a3 T9^{1/3} + a4 T9 + a5 T9^{5/3} + a6 ln T9)  [cm³ mol⁻¹ s⁻¹]
# (reaction index, set label, a0..a6). Labels: an06 Ando+ 2006, de04 Descouvemont+ 2004, gi17/go17 Gómez Iñesta+ 2017,
# cd08 Cyburt & Davids 2008, db18 Damone+ 2018.
REACLIB = [
    (1, "an06", [12.3687, 0.0, 0.0, -2.70618, 0.11718, -0.00312788, 0.469127]),
    (1, "an06", [10.7548, 0.0, 0.0, -2.30472, -0.887862, 0.137663, 0.0]),
    (1, "an06", [8.84688, 0.0, 0.0, -0.0102082, -0.0893959, 0.00696704, 1.0]),
    (2, "de04", [8.93525, 0.0, -3.7208, 0.198654, 0.0, 0.0, 0.333333]),
    (2, "de04", [7.52898, 0.0, -3.7208, 0.871782, 0.0, 0.0, -0.666667]),
    (3, "gi17", [19.0876, -0.00019002, -4.2292, 1.6932, -0.0855529, -1.35709e-25, -0.734513]),
    (4, "go17", [18.8052, 4.36209e-05, -4.32296, 1.91572, -0.081562, -3.28804e-22, -0.879518]),
    (5, "de04", [19.2762, 0.0, 0.0, 0.0438557, -0.201527, 0.0153433, 1.0]),
    (5, "de04", [20.3787, 0.0, 0.0, -0.332788, -0.700485, 0.0976521, 0.0]),
    (6, "de04", [39.3457, 0.0, -4.5244, -16.4028, 1.73103, -0.122966, 2.31304]),
    (6, "de04", [25.1794, 0.0, -4.5244, 0.350337, 0.58747, -8.84909, -0.666667]),
    (7, "de04", [41.2969, 0.0, -7.182, -17.1349, 1.36908, -0.0814423, 3.35395]),
    (7, "de04", [24.6839, 0.0, -7.182, 0.473288, 1.46847, -27.9603, -0.666667]),
    (8, "cd08", [17.7075, 0.0, -12.8271, -3.8126, 0.0942285, -0.00301018, 1.33333]),
    (8, "cd08", [15.6099, 0.0, -12.8271, -0.0308225, -0.654685, 0.0896331, -0.666667]),
    (9, "de04", [13.6162, 0.0, -8.0805, -0.217514, -0.114859, 0.0470043, -0.666667]),
    (10, "db18", [21.7899, 0.000728098, -0.30254, -0.3602, 0.17472, -0.0223, -0.4581]),
    (11, "de04", [21.8999, -26.1527, 0.0, 0.0, 0.0, 0.0, -1.5]),
    (11, "de04", [20.4438, 0.0, -8.4727, 0.297934, 0.0582335, -0.00413383, -0.666667]),
    (11, "de04", [14.2538, -4.478, 0.0, 0.0, 0.0, 0.0, -1.5]),
    (11, "de04", [11.9576, 0.0, -8.4727, 0.417943, 5.34565, -4.8684, -0.666667]),
]

# observed primordial abundances (cited constants, plotted as bands)
OBSERVED = {
    "Yp": (0.245, 0.003),  # PDG 2024 review of BBN (Fields, Molaro & Sarkar), He II/He I in metal-poor H II regions
    "DH": (2.527e-5, 0.030e-5),  # Cooke, Pettini & Steidel 2018, quasar absorption systems
    "LiH": (1.6e-10, 0.3e-10),  # Spite plateau of metal-poor halo stars (Sbordone et al. 2010)
}
ETA_PLANCK = (6.10e-10, 0.06e-10)  # from Planck 2018 Ω_b h² = 0.0224 (η₁₀ = 273.9 Ω_b h²)


def stoichiometry(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """R_{ri}, P_{ri}: how many of species i each reaction r consumes and produces, shape (Rx, N) each."""
    R = torch.zeros(len(REACTIONS), len(SPECIES), dtype=F64)
    P = torch.zeros_like(R)
    for r, (ins, outs) in enumerate(REACTIONS):
        for s in ins:
            R[r, SPECIES.index(s)] += 1
        for s in outs:
            P[r, SPECIES.index(s)] += 1
    return R.to(device), P.to(device)


def q_values(device: torch.device) -> torch.Tensor:
    """Q_r = Σ_i (R_ri − P_ri) Δ_i, energy released by each reaction (MeV), shape (Rx,). Q_0 = m_n − m_p."""
    R, P = stoichiometry(device)
    Q = (R - P) @ MASS_EXCESS.to(device)
    Q[0] = Q_NP
    return Q


def reaclib_table(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """The fit coefficients a_{sk} (S, 7) and the reaction index of every set (S,)."""
    a = torch.tensor([c for _, _, c in REACLIB], dtype=F64, device=device)
    idx = torch.tensor([r for r, _, _ in REACLIB], device=device)
    return a, idx


def eta_batch(n: int, eta_mark: float, device: torch.device) -> torch.Tensor:
    """η values: n log-spaced from 10⁻¹⁰ to 10⁻⁹, plus the marked value (Planck) as the last member, (n + 1,)."""
    grid = torch.logspace(-10, -9, n, dtype=F64)
    return torch.cat([grid, torch.tensor([eta_mark], dtype=F64)]).to(device)


def temperature_grid(T_start: float, T_end: float, n_steps: int, device: torch.device) -> torch.Tensor:
    """Photon temperatures in MeV, log-spaced from hot to cold, (S,): the integration variable is u = −ln T."""
    return torch.logspace(math.log10(T_start), math.log10(T_end), n_steps + 1, dtype=F64, device=device)


def initial_abundances(T0: float, B: int, device: torch.device) -> torch.Tensor:
    """Y_{bi} = n_i / n_baryon at T0: neutrons and protons in weak equilibrium, n/p = e^{−Q/T}; no nuclei yet."""
    Y = torch.zeros(B, len(SPECIES), dtype=F64, device=device)
    Y[:, 0] = 1 / (1 + math.exp(Q_NP / T0))
    Y[:, 1] = 1 - Y[:, 0]
    return Y


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/big_bang_nucleosynthesis/solve.py
"""Integrate the stiff network with an implicit BDF2 scheme in u = −ln T, all η values at once.

Each step solves Y_{n+1} − c_n = β h (dt/du) f(Y_{n+1}) by Newton's method; the Newton matrices of the whole η batch
are solved together with one torch.linalg.solve on a (B, N, N) tensor.
"""

import torch

from problems._common.progress import steps

from . import equation, tensors


def evolve(Y0: torch.Tensor, lam_f: torch.Tensor, lam_b: torch.Tensor, rate_u: torch.Tensor, u: torch.Tensor,
           newton: int = 4) -> torch.Tensor:  # fmt: skip
    """Y history (S + 1, B, N) from Y0 (B, N). lam_f/lam_b: (S + 1, B, Rx), rate_u = dt/du: (S + 1,), u: (S + 1,)."""
    R, P = tensors.stoichiometry(Y0.device)
    eye = torch.eye(Y0.shape[1], dtype=Y0.dtype, device=Y0.device)
    Ys = [Y0]
    for n in steps(len(u) - 1, "BDF2 steps"):
        h = float(u[n + 1] - u[n])
        if n == 0:  # first step: backward Euler
            c, beta = Ys[-1], 1.0
        else:  # BDF2: Y_{n+1} − (4/3) Y_n + (1/3) Y_{n−1} = (2/3) h f(Y_{n+1})
            c, beta = (4 * Ys[-1] - Ys[-2]) / 3, 2 / 3
        k = beta * h * float(rate_u[n + 1])
        Y = Ys[-1].clone()
        for _ in range(newton):
            f, J = equation.network(Y, lam_f[n + 1], lam_b[n + 1], R, P)
            G = Y - c - k * f  # residual (B, N)
            Y = (Y - torch.linalg.solve(eye - k * J, G[..., None])[..., 0]).clamp(min=0)
        Ys.append(Y)
    return torch.stack(Ys)


def run_bbn(etas: torch.Tensor, tau_n: float, n_nu: float, n_steps: int, device: torch.device,
            T_start: float = 3.0, T_end: float = 1e-3) -> dict:  # fmt: skip
    """The whole calculation for a batch of η: thermal history, rates, network, final abundances."""
    etas = etas.to(device=device, dtype=tensors.F64)
    T = tensors.temperature_grid(T_start, T_end, n_steps, device)  # (S + 1,) MeV, hot → cold
    lam_f, lam_b = equation.rate_coefficients(T, etas, tau_n)
    Y = evolve(tensors.initial_abundances(T_start, len(etas), device), lam_f, lam_b, equation.dt_du(T, n_nu), -T.log())
    final = equation.observables(Y[-1])
    return {"T": T, "t": equation.time_of(T, n_nu), "Y": Y, "etas": etas, **final}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/big_bang_nucleosynthesis/plot.py
"""Animate the light elements forming as the universe cools, then the Schramm plot of the yields against η."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .tensors import MASS_NUMBER, OBSERVED, SPECIES

COLOURS = ["#6b7280", "#111827", ACCENT[0], ACCENT[5], ACCENT[3], ACCENT[1], ACCENT[2], ACCENT[4]]


def animate(res: dict, eta_band: tuple[float, float], path: Path, n_grow: int = 50, n_sweep: int = 30) -> Path:
    """Left: mass fractions X_i = A_i Y_i against T for the last η of the batch (the marked value). Right: the
    final yields of the whole batch against η, revealed from low to high η, with the observed bands."""
    T, t = res["T"].cpu().numpy(), res["t"].cpu().numpy()
    X = (res["Y"][:, -1, :] * MASS_NUMBER.to(res["Y"])).cpu().numpy()  # (S, N) for the marked η
    etas = res["etas"].cpu().numpy()
    order = np.argsort(etas[:-1])
    eta = etas[:-1][order]
    curves = {k: res[k].cpu().numpy()[:-1][order] for k in ("Yp", "DH", "He3H", "LiH")}
    mark = {k: float(res[k][-1]) for k in ("Yp", "DH", "He3H", "LiH")}

    fig = plt.figure(figsize=(9.6, 5.2))
    gs = fig.add_gridspec(3, 2, width_ratios=[1.15, 1], hspace=0.12, wspace=0.28)
    ax = fig.add_subplot(gs[:, 0])
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(T[0], T[-1])
    ax.set_ylim(1e-12, 2)
    ax.set_xlabel("photon temperature T (MeV)   →  time")
    ax.set_ylabel("mass fraction  $X_i = A_i Y_i$")
    lines = [ax.plot([], [], lw=1.8, color=c, label=s)[0] for s, c in zip(SPECIES, COLOURS)]
    ax.legend(fontsize=7, ncol=2, loc="center left")
    clock = ax.text(0.97, 0.97, "", transform=ax.transAxes, ha="right", va="top", fontsize=8,
                    bbox={"fc": "white", "ec": "#d1d5db"})  # fmt: skip
    ax.set_title(f"η = {etas[-1] * 1e10:.2f} × 10⁻¹⁰", fontsize=10)

    ax_y, ax_d, ax_l = (fig.add_subplot(gs[k, 1]) for k in range(3))
    specs = [
        (ax_y, [("Yp", ACCENT[1], "$Y_p$")], "Yp", (0.20, 0.27), False),
        (ax_d, [("DH", ACCENT[0], "D/H"), ("He3H", ACCENT[3], "³He/H")], "DH", (3e-6, 1e-3), True),
        (ax_l, [("LiH", ACCENT[2], "⁷Li/H")], "LiH", (5e-11, 2e-9), True),
    ]
    sweep = []
    for a, keys, obs, ylim, logy in specs:
        a.set_xscale("log")
        a.set_xlim(eta[0], eta[-1])
        a.set_ylim(*ylim)
        if logy:
            a.set_yscale("log")
        a.axvspan(eta_band[0] - eta_band[1], eta_band[0] + eta_band[1], color="#9ca3af", alpha=0.45, lw=0)
        v, e = OBSERVED[obs]
        a.axhspan(v - 2 * e, v + 2 * e, color=keys[0][1], alpha=0.15, lw=0)
        for k, c, lab in keys:
            sweep.append((a.plot([], [], lw=2, color=c, label=lab)[0], k))
        a.legend(fontsize=7, loc="upper right" if obs == "Yp" else "upper left", ncol=2)
        a.tick_params(labelsize=7)
        if a is not ax_l:
            a.set_xticklabels([])
    ax_y.set_title("Schramm plot: yields vs η (bands: observed ±2σ, grey: Planck)", fontsize=8)
    ax_l.set_xlabel("baryon-to-photon ratio η", fontsize=8)
    li_note = ax_l.text(0.03, 0.08, "", transform=ax_l.transAxes, fontsize=7, color="#b91c1c")

    k_grow = [round(j * (len(T) - 1) / (n_grow - 1)) for j in range(n_grow)]  # temperature steps to show

    def update(frame: int):
        k = k_grow[min(frame, n_grow - 1)]
        for i, line in enumerate(lines):
            line.set_data(T[: k + 1 : 2], np.maximum(X[: k + 1 : 2, i], 1e-30))  # every 2nd step is enough to draw
        clock.set_text(f"T = {T[k] * 1e3:.4g} keV\nt = {t[k]:.3g} s")
        m = 0 if frame < n_grow else round(len(eta) * min(1.0, (frame - n_grow + 1) / n_sweep))
        for line, key in sweep:
            line.set_data(eta[:m], curves[key][:m])
        if frame >= n_grow + n_sweep - 1:
            li_note.set_text(
                f"Planck η: Y_p = {mark['Yp']:.4f}, D/H = {mark['DH']:.2e}\n"
                f"⁷Li/H = {mark['LiH']:.1e} = {mark['LiH'] / OBSERVED['LiH'][0]:.1f}× observed: lithium problem"
            )

    fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.1)
    return save_gif(fig, update, frame_indices(n_grow + n_sweep + 10), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_eta': 24,
    'eta10': 6.1,
    'tau_n': 878.4,
    'n_nu': 3.0,
    'n_steps': 1200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.big_bang_nucleosynthesis")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.big_bang_nucleosynthesis.main")
problem_dir = Path("problems/cosmology/big_bang_nucleosynthesis")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
